習題一：使用random模組，產生出一個介於0到1之間的小數$r$，使用while迴圈，計算出$r$的前二十位二進位表示$r = 0.b_1b_2b_3\cdots {b_{20}}_2$。輸出格式如附圖Exercise20261015.png

In [12]:
import random

r = random.random()
fraction = r
bits = ""

count = 0
while count < 20:
    fraction *=2
    bit = int(fraction)
    bits += str(bit)
    fraction -= bit
    count += 1

print(f"假如r = {r}")
print(f"r = 0.{bits}₂")    

假如r = 0.24742975757673114
r = 0.00111111010101111000₂


Python 的 `int` 型別採用 Arbitrary-Precision Integer（任意精度整數）設計，能夠精確表示任意大小的整數，其實作方式可參考 CPython 的 `PyLongObject`。與 C/C++ 的 `int` 或 `long long` 不同，Python 的整數不受固定 32 位元或64 位元限制，而是會依據數值大小自動配置更多記憶體，因此只要記憶體足夠，便能表示極大的整數。例如：
```python
n = 10**68
print(n)
```
上述程式可正常執行，表示 Python 能夠精確儲存$10^{68}$，即中文數字中的「無量大數」。


可利用下列程式觀察不同大小整數所需的位元數（bit）與記憶體空間：

```python
import sys
n = 10**68
print(f"bit_length = {n.bit_length()}")
print(f"memory = {sys.getsizeof(n)} bytes")
```

並查閱官方文件，了解：

- `bit_length()`
- `sys.getsizeof()`
- `PyLongObject`

三者之間的關係。

習題二：輸入一個正整數 \(n\)，其中$1 \le n \le 10^{68}$，請將 \(n\) 轉換為中文大寫數字表示法。需支援下列數字單位：萬、億、兆、京、垓、秭、穰、溝、澗、正、載、極、恆河沙、阿僧祇、那由他、不可思議、無量大數。比如輸入31415926，輸出參仟壹佰肆拾壹萬伍仟玖佰貳拾陸。

提示：思考利用字典，建立「數字」與「中文大寫數字」的對應關係。

In [1]:

digits = {
    0: '零',
    1: '壹',
    2: '貳',
    3: '參',
    4: '肆',
    5: '伍',
    6: '陸',
    7: '柒',
    8: '捌',
    9: '玖'
}

small_units = ['', '拾', '佰', '仟']

big_units = [
    '', '萬', '億', '兆', '京', '垓',
    '秭', '穰', '溝', '澗', '正', '載',
    '極', '恆河沙', '阿僧祇', '那由他',
    '不可思議', '無量大數'
]


def convert_group(number):
    """將 0～9999 的數字轉成中文大寫，不加大單位。"""
    result = ''
    zero_pending = False

    # 固定成四位數，方便依序處理仟、佰、拾、個
    number_str = str(number).zfill(4)

    for i, char in enumerate(number_str):
        d = int(char)
        unit = small_units[3 - i]

        if d == 0:
            # 只有前面已有有效數字時，才需要考慮補零
            if result:
                zero_pending = True
        else:
            # 中間有零，而且後面又出現非零數字
            if zero_pending:
                result += digits[0]
                zero_pending = False

            result += digits[d] + unit

    return result


def to_chinese_upper(n):
    """將正整數轉換成中文大寫數字。"""
    if not isinstance(n, int) or isinstance(n, bool):
        raise ValueError('輸入必須是整數')

    if not 1 <= n <= 10**68:
        raise ValueError('輸入範圍必須是 1 到 10**68')

    # 從個位起，每四位切成一組
    groups = []
    while n > 0:
        groups.append(n % 10000)
        n //= 10000

    result = ''
    zero_group = False

    # 由最高位組別往最低位組別處理
    for i in range(len(groups) - 1, -1, -1):
        group = groups[i]

        if group == 0:
            zero_group = True
            continue

        # 非第一個有效組別，且中間有空組或目前不足四位
        if result and (zero_group or group < 1000):
            result += '零'

        result += convert_group(group) + big_units[i]
        zero_group = False

    return result


n = int(input('請輸入正整數 n：'))
print(to_chinese_upper(n))


貳拾伍萬參仟伍佰伍拾伍


習題三：請勿使用for、range，而是利用iterable、iter()、next()、while、StopIteration來設計九九乘法表。

In [2]:
numbers = iter([1, 2, 3, 4, 5, 6, 7, 8, 9])

try:
    while True:
        i = next(numbers)
        numbers2 = iter([1, 2, 3, 4, 5, 6, 7, 8, 9])

        try:
            while True:
                j = next(numbers2)
                print(f'{i}*{j}={i*j}', end='\t')
        except StopIteration:
            print()

except StopIteration:
    pass

1*1=1	1*2=2	1*3=3	1*4=4	1*5=5	1*6=6	1*7=7	1*8=8	1*9=9	
2*1=2	2*2=4	2*3=6	2*4=8	2*5=10	2*6=12	2*7=14	2*8=16	2*9=18	
3*1=3	3*2=6	3*3=9	3*4=12	3*5=15	3*6=18	3*7=21	3*8=24	3*9=27	
4*1=4	4*2=8	4*3=12	4*4=16	4*5=20	4*6=24	4*7=28	4*8=32	4*9=36	
5*1=5	5*2=10	5*3=15	5*4=20	5*5=25	5*6=30	5*7=35	5*8=40	5*9=45	
6*1=6	6*2=12	6*3=18	6*4=24	6*5=30	6*6=36	6*7=42	6*8=48	6*9=54	
7*1=7	7*2=14	7*3=21	7*4=28	7*5=35	7*6=42	7*7=49	7*8=56	7*9=63	
8*1=8	8*2=16	8*3=24	8*4=32	8*5=40	8*6=48	8*7=56	8*8=64	8*9=72	
9*1=9	9*2=18	9*3=27	9*4=36	9*5=45	9*6=54	9*7=63	9*8=72	9*9=81	
